In [1]:
import torch
torch.__version__

'2.2.2'

In [2]:
import torch
import torchvision
model = torchvision.models.resnet50()

###  Important Note: `torch.compile()` & Python Version Compatibility
* **Issue:** Encountered `RuntimeError: Dynamo is not supported on Python 3.12+` when running `torch.compile()`.
* **Cause:** TorchDynamo (PyTorch's underlying compilation engine) does not fully support Python 3.12 or higher yet.
* **Solution:** Switch to a kernel/environment using a fully supported Python version (e.g., Python 3.10 or 3.11) and reinstall `torch` and `torchvision`.

In [3]:
import torch
import torchvision
model = torchvision.models.resnet50() #this could be any model
compiled_model = torch.compile(model) #this going to do some optimization steps behind the scenes

#training code


#testing code



In [4]:
import torch

# التحقق مما إذا كان تسريع الـ Mac (MPS) مدعوماً ومتاحاً
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Mac GPU (MPS) - PyTorch acceleration is available!")
else:
    device = torch.device("cpu")
    print("MPS device not found. Using CPU instead.")

Using Mac GPU (MPS) - PyTorch acceleration is available!


## Globally set devices
Previously, we've set the device of our tensors/models using `.to(device)`:

* `tensor.to(device)`
* `model.to(device)`

But in PyTorch 2.0, it's possible to set the device with a context manager as well as a global device.

### 1. Setting Device with a Context Manager
Using `with torch.device(device):`, any PyTorch objects or tensors created **inside** the code block will automatically be allocated to the specified device.

### 2. Setting the Device Globally
Using `torch.set_default_device(device)`, the default device is changed globally for everything created from that point forward in the script/notebook.

In [5]:
import torch
#context manager version
# Set the device for Mac GPU (MPS)
device = "mps" if torch.backends.mps.is_available() else "cpu"

# Set the device with context manager (requires PyTorch 2.x+)
with torch.device(device): 
    # All tensors or PyTorch objects created in the context manager will be on the target device
    layer = torch.nn.Linear(20, 30)
    print(f"Layer weights are on device: {layer.weight.device}")
    print(f"Layer creating data on device: {layer(torch.randn(128, 20)).device}")

Layer weights are on device: mps:0
Layer creating data on device: mps:0


In [6]:
import torch
#global manager version
# Set the device for Mac GPU (MPS)
device = "mps" if torch.backends.mps.is_available() else "cpu"

# Set the device globally (requires PyTorch 2.x+)
torch.set_default_device(device)

# All tensors or PyTorch objects created from here on out will be on the target device without using .to()
layer = torch.nn.Linear(20, 30)
print(f"Layer weights are on device: {layer.weight.device}")
print(f"Layer creating data on device: {layer(torch.randn(128, 20)).device}")

# Reset the default device back to CPU so it does not leak into the rest of the notebook
# (a global MPS default breaks DataLoader(shuffle=True): "Expected a 'mps:0' generator device but found 'cpu'")
torch.set_default_device("cpu")

Layer weights are on device: mps:0
Layer creating data on device: mps:0


##  Setting up the experiments

Time to test speed!

To keep things simple, we'll run 4 experiments:

* **Model:** ResNet50 from `torchvision`
* **Data:** CIFAR10 from `torchvision`
* **Epochs:** 5 (single run) and 3x5 (multi run)
* **Batch size:** 128 (note: you may want to change this depending on the amount of memory your GPU has, this tutorial focuses on an A100)
* **Image size:** 224 (note: you may want to adjust this given the amount of GPU memory you have)

**Objective:** compare training/testing speed **with** and **without** `torch.compile()`

In [7]:
#create model weights and transforms
model_weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 #we can use default also
transforms = model_weights.transforms()
transforms

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [8]:
#create the model
model = torchvision.models.resnet50(weights = model_weights)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [9]:
#count the number of parameters in the model
total_params = sum(
    param.numel() for param in model.parameters() #count all parameters
)
total_params

25557032

**Note:** PyTorch 2.0 *relative* speedups will be most noticeable when as much of the GPU as possible is being used. This means a larger model (more trainable parameters) may take longer to train on the whole but will be relatively faster. E.g. a model with 1M parameters may take ~10 minutes to train but a model with 25M parameters may take ~20 minutes to train.

In [10]:
import torchvision
def create_model(num_classes = 10): #num_classes = 10 depends on the data we are working on
    model_weights = torchvision.models.ResNet50_Weights.DEFAULT
    transforms = model_weights.transforms()
    model = torchvision.models.resnet50(weights = model_weights)
    
    #adjust the head layer to suit our number of classes
    model.fc = torch.nn.Linear(in_features = 2048, out_features = num_classes)
    
    return model, transforms
model, transforms = create_model()

In [11]:
model, transforms

(ResNet(
   (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
   (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (relu): ReLU(inplace=True)
   (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
   (layer1): Sequential(
     (0): Bottleneck(
       (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
       (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
       (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
       (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (relu): ReLU(inplace=True)
       (downsample): Sequential(
         (0): Conv2d(64, 256, kernel_size=(1,

###  Why Increase GPU Workload? (PyTorch 2.0 & Speedups)

Modern GPUs are extremely fast. To experience the **true performance benefits and relative speedups** (e.g., from features like `torch.compile()`), you need to utilize as much of the GPU memory and processing power as possible. 

If the workload is too small, the GPU sits mostly idle, and overhead takes up most of the time.

#### Key Factors to Scale Up:

1. **Increase the Batch Size**
   * *Concept:* Instead of processing a small batch (e.g., 32), push it as high as your GPU memory allows (e.g., 128, 256, 512).
   * *Why:* Keeps the GPU heavily utilized and parallelizes the computations.

2. **Increase Data/Image Size**
   * *Concept:* Instead of tiny images (e.g., $32 \times 32$), scale up to larger resolutions (e.g., $224 \times 224$ or $336 \times 336$).
   * *Why:* Larger matrices and richer inputs demand more computational work from the hardware.

3. **Increase Model Size**
   * *Concept:* Use models with more trainable parameters (e.g., millions of parameters like ResNet50) rather than tiny micro-models.
   * *Why:* Heavier, deeper architectures feed the GPU cores enough operations to show a noticeable difference in training/testing execution time.

4. **Decreasing Data Transfer**
   * *Concept:* Minimize the constant back-and-forth movement of data between system memory (RAM) and GPU memory (VRAM).
   * *Why:* Bandwidth costs and slow data transfers stall the GPU; keeping data flowing efficiently ensures the GPU spends its time computing rather than waiting (preventing GPU starvation).

> **Rule of Thumb:** Maximize your GPU memory utilization for optimal throughput and to clearly measure performance optimizations.(use as much of the GPU as you can)

In [12]:
import psutil

# Get total system RAM in GB
total_ram = round(psutil.virtual_memory().total * 1e-9, 2)
print(f"Total Mac RAM: {total_ram} GB")

Total Mac RAM: 17.18 GB


In [13]:
import psutil

# Get total and available system RAM in GB
total_ram = round(psutil.virtual_memory().total * 1e-9, 2)
available_ram = round(psutil.virtual_memory().available * 1e-9, 2)

print(f"Total Mac RAM: {total_ram} GB")
print(f"Available Mac RAM: {available_ram} GB")

Total Mac RAM: 17.18 GB
Available Mac RAM: 5.89 GB


 Set batch size based on available memory logic:
 - If GPU free memory >= 16GB -> set BATCH_SIZE = 128
 - If GPU free memory < 16GB  -> set BATCH_SIZE = 32
 Note: On Mac, we check system available RAM due to Unified Memory architecture.

In [14]:
import psutil

# Set batch size and image size dynamically based on available system memory (Mac Unified Memory)
available_ram_gb = round(psutil.virtual_memory().available * 1e-9, 3)

if available_ram_gb >= 16:
    BATCH_SIZE = 128
    IMAGE_SIZE = 224
    print(f"Available memory is {available_ram_gb} GB, using batch size of {BATCH_SIZE} and image size of {IMAGE_SIZE}")
else:
    BATCH_SIZE = 32
    IMAGE_SIZE = 128
    print(f"Available memory is {available_ram_gb} GB, using batch size of {BATCH_SIZE} and image size of {IMAGE_SIZE}")

Available memory is 5.892 GB, using batch size of 32 and image size of 128


In [15]:
transforms #we must update the transform to use this image size

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [16]:
transforms.crop_size = IMAGE_SIZE
transforms.resize_size = IMAGE_SIZE
print(f"the updated transforms:{transforms}")

the updated transforms:ImageClassification(
    crop_size=128
    resize_size=128
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [17]:
# get the dataset
train_data = torchvision.datasets.CIFAR10(root = '.', #where to store data
                                          train = True, #do we want training data?
                                          download = True,
                                          transform = transforms)

test_data = torchvision.datasets.CIFAR10(root = '.', #where to store data
                                          train = False, #do we want training data?
                                          download = True,
                                          transform = transforms)

Files already downloaded and verified


Files already downloaded and verified


In [18]:
len(train_data), len(test_data)

(50000, 10000)

In [19]:
from torch.utils.data import DataLoader
import os

NUM_WORKERS = 0

train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=NUM_WORKERS)

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             shuffle=False,
                             num_workers=NUM_WORKERS)

# Print details:
print(f"Train dataloader num batches: {len(train_dataloader)} of batch size: {BATCH_SIZE}")
print(f"Test dataloader num batches: {len(test_dataloader)} of batch size: {BATCH_SIZE}")
print(f"Using num workers to load data (more is generally better): {NUM_WORKERS}")

Train dataloader num batches: 1563 of batch size: 32
Test dataloader num batches: 313 of batch size: 32
Using num workers to load data (more is generally better): 0


In [20]:
50000/32


1562.5

In [21]:
10000/32

312.5

In [22]:
import time
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(epoch: int,
               model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device,
               disable_progress_bar: bool = False) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).

    Args:
      epoch: Integer representing the current epoch number (e.g. 0).
      model: A PyTorch model to be trained.
      dataloader: A DataLoader instance for the model to be trained on.
      loss_fn: A PyTorch loss function to minimize.
      optimizer: A PyTorch optimizer to help minimize the loss function.
      device: A target device to compute on (e.g. "cuda" or "cpu").
      disable_progress_bar: Boolean whether or not to disable tqdm progress bar.

    Returns:
      A tuple of training loss and training accuracy metrics.
      In the form (train_loss, train_accuracy). For example:
      (0.1112, 0.8743)
    """
    # Put model in train mode
    model.train()
    
    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0
    
    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)
        
        # 1. Forward pass
        y_pred = model(X)
        
        # 2. Calculate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        # 3. Optimizer zero grad
        optimizer.zero_grad()
        
        # 4. Loss backward
        loss.backward()
        
        # 5. Optimizer step
        optimizer.step()
        
        # Calculate accuracy metric
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y)
        
    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [23]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """Computes loss and accuracy for a test PyTorch model.

    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Args:
      model: A PyTorch model to be tested.
      dataloader: A DataLoader instance for the model to be tested on.
      loss_fn: A PyTorch loss function to calculate loss on the test data.
      device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
      A tuple of testing loss and testing accuracy metrics.
      In the form (test_loss, test_accuracy). For example:
      (0.0223, 0.8985)
    """
    # Put model in eval mode
    model.eval()
    
    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0
    
    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)
            
            # 1. Forward pass
            test_pred_logits = model(X)
            
            # 2. Calculate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            # Calculate accuracy
            test_pred_class = torch.argmax(torch.softmax(test_pred_logits, dim=1), dim=1)
            test_acc += (test_pred_class == y).sum().item() / len(y)
            
        # Adjust metrics to get average loss and accuracy per batch
        test_loss = test_loss / len(dataloader)
        test_acc = test_acc / len(dataloader)
        return test_loss, test_acc

In [24]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          disable_progress_bar: bool = False) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics.

    Args:
      model: A PyTorch model to be trained and tested.
      train_dataloader: A DataLoader instance for the model to be trained on.
      test_dataloader: A DataLoader instance for the model to be tested on.
      optimizer: A PyTorch optimizer to help minimize the loss function.
      loss_fn: A PyTorch loss function to calculate loss on both datasets.
      epochs: An integer indicating how many epochs to train for.
      device: A target device to compute on (e.g. "cuda" or "cpu").
      disable_progress_bar: Boolean whether or not to disable tqdm progress bar.

    Returns:
      A dictionary of training and testing loss as well as training and
      testing accuracy metrics. Each metric has a value in a list for 
      each epoch.
      In the form: {r"train_loss": [...],
                    "train_acc": [...],
                    "test_loss": [...],
                    "test_acc": [...]} 
      For example for training for epochs=2: 
               {r"train_loss": [2.0616, 1.0516],
                "train_acc": [0.3945, 0.3945],
                "test_loss": [1.2641, 1.5706],
                "test_acc": [0.3400, 0.2973]}
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": [],
               "train_epoch_time": [],
               "test_epoch_time": []
    }
    
    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs), disable=disable_progress_bar):
        
        # Perform training step and time it
        train_epoch_start_time = time.time()
        train_loss, train_acc = train_step(epoch=epoch,
                                           model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device,
                                           disable_progress_bar=disable_progress_bar)
        train_epoch_end_time = time.time()
        train_epoch_time = train_epoch_end_time - train_epoch_start_time
        
        # Perform testing step and time it
        test_epoch_start_time = time.time()
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)
        test_epoch_end_time = time.time()
        test_epoch_time = test_epoch_end_time - test_epoch_start_time
        
        # Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f} | "
            f"train_epoch_time: {train_epoch_time:.4f}s | "
            f"test_epoch_time: {test_epoch_time:.4f}s"
        )
        
        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
        results["train_epoch_time"].append(train_epoch_time)
        results["test_epoch_time"].append(test_epoch_time)

    # Return the filled results at the end of the epochs
    return results

In [25]:
import torch

# تحديد الجهاز ليعمل على كارت الشاشة الداخلي للماك (MPS) أو الـ CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [ ]:
#Experiment 1 : single run without torch.compile() for 5 epochs

#create a model
model, _ = create_model()
model.to(device)

#create a loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.003)

#train the model and track the results
single_run_no_compile_result = train(model = model,
          train_dataloader = train_dataloader,
          test_dataloader = test_dataloader,
          optimizer = optimizer,
          loss_fn = loss_fn,
          epochs = 5,
          device = device)

### Note
I couldn't finish the experiments in this section because my laptop doesn't have an NVIDIA GPU (2018 Intel MacBook Pro).
* `torch.compile()` didn't work on Python 3.12, so I had to use a Python 3.11 kernel, and on my Mac it only works on the CPU, not on the GPU (MPS).
* Training ResNet50 on my laptop takes a very long time, so I stopped after setting up Experiment 1.
* To see the real speedups from `torch.compile()`, this notebook should be run on an NVIDIA GPU (e.g. Google Colab).